# Burger recipes rated by the same customers

**One-way Repeated Measures Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A burger chain asked five regular customers to taste three recipes for a new burger and rate each from 0 to 10. Every customer tasted all three.

**The question.** Do the recipes differ, once we allow for some customers simply being more generous?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/one_way_anova_repeated.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Customer,Recipe A,Recipe B,Recipe C
1.0,8.0,8.1,7.2
2.0,6.6,6.5,5.3
3.0,7.4,7.9,7.0
4.0,6.6,7.1,6.2
5.0,7.6,7.7,6.7

These cells save `eta_squared_interval.py`, `sphericity.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile eta_squared_interval.py
"""The confidence interval for partial eta squared, from the noncentral F distribution.

SciPy has the noncentral F (stats.ncf) but no function that inverts it for an interval,
so the inversion is written out (Steiger 2004). The interval is found for the
noncentrality λ and converted to partial eta squared as λ / (λ + df1 + df2 + 1).
"""

import numpy as np
from scipy import optimize, stats


def _limit(f, df1, df2, q):
    """The λ ≥ 0 under which the observed F sits at the q quantile of F(df1, df2, λ).

    0 when even λ = 0 cannot put it there: an F too small for the interval to leave 0.
    The cdf at F falls as λ rises, so the bracket opens outwards from F·df1, doubling.
    """
    def gap(lam):
        cdf = stats.f.cdf(f, df1, df2) if lam == 0 else stats.ncf.cdf(f, df1, df2, lam)
        return cdf - q

    if gap(0.0) <= 0:
        return 0.0
    start = max(f * df1, 1.0)
    step = 1.0
    while gap(max(start - step, 0.0)) <= 0:
        step *= 2
    low = max(start - step, 0.0)
    step = 1.0
    while gap(start + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, start + step, xtol=1e-14,
                           rtol=4 * np.finfo(float).eps, maxiter=500)


def partial_eta_squared_interval(f, df1, df2, level):
    """(lower, upper): the level interval for partial eta squared (Steiger 2004)."""
    total = df1 + df2 + 1
    lower = _limit(f, df1, df2, (1 + level) / 2)
    upper = _limit(f, df1, df2, (1 - level) / 2)
    return lower / (lower + total), upper / (upper + total)

In [ ]:
%%writefile sphericity.py
"""Mauchly's test of sphericity and the epsilon corrections for repeated measures.

statsmodels' AnovaRM prints only the uncorrected F test and SciPy has neither, so both
are written out. Each takes the covariance matrix of one within-subject effect's
orthonormal contrast scores -- the residuals of the between-subject model when there is
one -- which is all that sphericity is about.
"""

import numpy as np
from scipy import stats


def mauchly(covariance, df):
    """Mauchly's W, its chi-square approximation, the chi-square df and P.

    ``df`` is the error df of the covariance matrix (subjects minus groups). W is 1 when
    the contrasts are uncorrelated with equal variances, which is what sphericity means.
    With one contrast -- a two-level factor -- there is nothing to test, so W = 1 and P
    is undefined.
    """
    p = covariance.shape[0]
    if p < 2:
        return 1.0, 0.0, 0, np.nan
    w = np.linalg.det(covariance) / (np.trace(covariance) / p) ** p
    correction = 1 - (2 * p**2 + p + 2) / (6 * p * df)
    chi_square = -df * correction * np.log(w)
    chi_df = p * (p + 1) // 2 - 1
    return w, chi_square, chi_df, stats.chi2.sf(chi_square, chi_df)


def epsilons(covariance, n, groups):
    """(lower bound, Greenhouse-Geisser, Huynh-Feldt-Lecoutre, Chi-Muller) for one effect.

    Epsilon is 1 under sphericity and 1/p at worst. Huynh-Feldt's estimate uses
    Lecoutre's (1991) correction, (N - g + 1) in the numerator, and can exceed 1; the
    table caps it at 1 when it multiplies the degrees of freedom. Chi-Muller shrinks it,
    since it runs high in small samples, by a multiple fixed by the design alone,
    (m - 1)(m - 2) / m^2 with m = p (N - g - 1), and floors it at the lower bound
    (Chi et al. 2012; SAS's UEPSDEF=CM).
    """
    p = covariance.shape[0]
    lower = 1 / p
    gg = np.trace(covariance) ** 2 / (p * np.trace(covariance @ covariance))
    gg = min(max(gg, lower), 1.0)  # rounding can step a hair outside [1/p, 1]
    df = n - groups
    hfl = ((df + 1) * p * gg - 2) / (p * (df - p * gg))
    m = p * (df - 1)
    cm = max(hfl * (m - 1) * (m - 2) / m**2, lower) if m > 0 else lower
    return lower, gg, hfl, cm

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from scipy.linalg import helmert
from statsmodels.stats.multitest import multipletests
from sphericity import epsilons, mauchly

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per subject, one column per condition. A subject with a blank in any
# condition is left out of every condition: each subject is compared only with
# themselves, so a subject needs every measurement.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["Recipe A", "Recipe B", "Recipe C"]
factor_name = "Recipe" or "Factor A"
subject_name = "Customer" or "Subject"
y = data[levels].apply(pd.to_numeric, errors="coerce").dropna().to_numpy(float)
n, k = y.shape  # subjects, conditions

## Descriptive statistics

In [ ]:
means = y.mean(axis=0)
sds = y.std(axis=0, ddof=1)
ses = sds / np.sqrt(n)
descriptive = pd.DataFrame(
    {"n": n, "mean": means, "SD": sds, "mean - SD": means - sds, "mean + SD": means + sds,
     "SE": ses, "mean - SE": means - ses, "mean + SE": means + ses},
    index=levels,
)
print(descriptive)

## Mauchly's test of sphericity

In [ ]:
# Sphericity concerns the k - 1 contrasts among the conditions. Helmert contrasts
# scaled to length 1 serve (any orthonormal set gives the same answer): under
# sphericity their covariance across subjects is a multiple of the identity.
contrasts = helmert(k).T  # k x (k - 1)
scores = y @ contrasts
centred = scores - scores.mean(axis=0)
covariance = centred.T @ centred / (n - 1)
w, chi_square, chi_df, chi_p = mauchly(covariance, n - 1)
lower, gg, hfl, cm = epsilons(covariance, n, groups=1)
sphericity = pd.DataFrame(
    [[w, chi_square, chi_df, chi_p, lower, gg, hfl, cm]], index=[factor_name],
    columns=["W", "chi-square", "df", "P", "lower bound", "Greenhouse-Geisser",
             "Huynh-Feldt-Lecoutre", "Chi-Muller"],
)
print()
print(sphericity.to_string())

## Analysis of variance table

In [ ]:
# The total variation splits into conditions, subjects and what is left (the error).
# Subject-to-subject differences leave the error term, which is the point of measuring
# the same subjects again.
grand = y.mean()
ss_condition = n * ((y.mean(axis=0) - grand) ** 2).sum()
ss_subject = k * ((y.mean(axis=1) - grand) ** 2).sum()
ss_total = ((y - grand) ** 2).sum()
ss_error = ss_total - ss_condition - ss_subject
df_condition, df_subject = k - 1, n - 1
df_error = df_condition * df_subject
f_value = (ss_condition / df_condition) / (ss_error / df_error)

# Each correction multiplies both degrees of freedom by its epsilon (Huynh-Feldt capped
# at 1). F never changes; only the distribution it is judged against does.
corrections = {"Sphericity assumed": 1.0, "Greenhouse-Geisser": gg,
               "Huynh-Feldt-Lecoutre": min(hfl, 1.0), "Chi-Muller": min(cm, 1.0),
               "Lower bound": lower}
rows = []
for correction, epsilon in corrections.items():
    df1, df2 = df_condition * epsilon, df_error * epsilon
    rows.append([factor_name, correction, ss_condition, df1, ss_condition / df1, f_value,
                 stats.f.sf(f_value, df1, df2)])
for correction, epsilon in corrections.items():
    df2 = df_error * epsilon
    rows.append([f"Error ({factor_name})", correction, ss_error, df2, ss_error / df2,
                 np.nan, np.nan])
rows.append([f"{subject_name} (error)", "", ss_subject, df_subject, ss_subject / df_subject,
             np.nan, np.nan])
rows.append(["Total", "", ss_total, n * k - 1, np.nan, np.nan, np.nan])
anova = pd.DataFrame(rows, columns=["source", "correction", "SS", "df", "MS", "F", "P"])
print()
print(anova.to_string(index=False))

## Multiple comparisons

In [ ]:
# Every pair of conditions, with a t statistic on the error term of the table above,
# then each P value adjusted for the number of pairs by the chosen procedures.
# Any of "Bonferroni", "Sidak" and "Holm", in the order to print them; [] for none.
procedures = ["Sidak"]
ms_error = ss_error / df_error
se_difference = np.sqrt(2 * ms_error / n)
pairs = list(combinations(range(k), 2))
difference = np.array([means[i] - means[j] for i, j in pairs])
t_value = difference / se_difference
p_raw = 2 * stats.t.sf(np.abs(t_value), df_error)
rows = []
for procedure in procedures:
    p_adjusted = multipletests(p_raw, method=procedure.lower())[1]
    for (i, j), d, t, p in zip(pairs, difference, t_value, p_adjusted):
        rows.append([procedure, levels[i], levels[j], means[i], means[j], d, se_difference, t, p])
comparisons = pd.DataFrame(rows, columns=["method", "level 1", "level 2", "mean 1", "mean 2",
                                          "difference", "SE", "t", "P"])
print()
print(comparisons.to_string(index=False))

## Effect sizes

In [ ]:
# Partial eta² sets the conditions against their own error: SS conditions / (SS
# conditions + SS error). It carries the 95% interval that inverts the noncentral F at
# the sphericity-assumed F (Steiger 2004), written out in eta_squared_interval.py.
# Generalised eta² puts the subjects back in the denominator -- SS conditions / (SS
# conditions + SS subjects + SS error) -- so it compares with an eta² measured between
# subjects (Olejnik & Algina 2003, the factor treated as manipulated).
from eta_squared_interval import partial_eta_squared_interval


def size(value):
    """Cohen's (1988) reading of an eta²: 0.01 small, 0.06 medium, 0.14 large."""
    return ["negligible", "small", "medium", "large"][sum(value >= s for s in (0.01, 0.06, 0.14))]


lower, upper = partial_eta_squared_interval(f_value, df_condition, df_error, 0.95)
effect_sizes = pd.DataFrame(
    [[ss_condition / (ss_condition + ss_error), lower, upper],
     [ss_condition / (ss_condition + ss_subject + ss_error), np.nan, np.nan]],
    index=[f"{factor_name}: partial eta²", f"{factor_name}: generalised eta²"],
    columns=["value", "lower", "upper"],
)
effect_sizes["size"] = effect_sizes["value"].map(size)
print()
print(effect_sizes)

## What the result says

Each customer rated all three recipes, so the recipes are compared inside each customer. This removes the fact that some customers give high marks to everything.

Mauchly's test checks sphericity, the assumption that the differences between recipes vary equally. Here P = 0.037, so the assumption fails and a corrected row should be read. With the Greenhouse-Geisser correction, F = 34.2 and P = 0.003. The P value is the chance of a result at least this extreme if the recipes were equal.

The Sidak comparisons show where. Recipe C (mean 6.48) is rated below A (7.24) and B (7.46), by 0.76 and 0.98 points. For A against B, P = 0.307, so they cannot be told apart. Drop recipe C.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.